# Add Tracking-Derived Slot Support

Purpose: add player-location features at the origin shot moment.

This notebook starts from `origin_shot_sequences.parquet`, which has one row per non-overlapping shot-created chain.

The key tracking question is:

> At the moment the origin shot was released, how many **identified** attacking and defending skaters were observed in the slot?

These are observed tracking features, not a full on-ice census. A player who is not identified in the event-frame tracking data is not counted as a skater in the slot.

This notebook does not answer the final outside-shot question. It creates tracking-derived features for the later strength/context and analysis notebooks.

## 1. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw" / "halo_2026"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Raw data exists:", DATA_RAW.exists())
print("Processed data exists:", DATA_PROCESSED.exists())

Project root: c:\Users\rinal\hockey-analytics\outside-shot-value
Raw data exists: True
Processed data exists: True


## 2. Load Inputs

Inputs:

- `origin_shot_sequences.parquet`: one row per non-overlapping shot-created chain
- `shot_value_base.parquet`: evaluated chances with origin context
- `events.parquet`: origin event coordinates and orientation
- `tracking.parquet`: player coordinates at event moments
- `players.parquet`: player position metadata for goalie exclusion

Tracking is joined at the origin event, not the evaluated chance event.

In [3]:
origin_sequences = pd.read_parquet(DATA_PROCESSED / "origin_shot_sequences.parquet")
shot_value_base = pd.read_parquet(DATA_PROCESSED / "shot_value_base.parquet")

events = pd.read_parquet(DATA_RAW / "events.parquet")
tracking = pd.read_parquet(DATA_RAW / "tracking.parquet")
players = pd.read_parquet(DATA_RAW / "players.parquet")

events["period_time"] = pd.to_numeric(events["period_time"], errors="coerce")

print("origin_sequences:", origin_sequences.shape)
print("shot_value_base:", shot_value_base.shape)
print("events:", events.shape)
print("tracking:", tracking.shape)
print("players:", players.shape)

origin_sequences: (48673, 43)
shot_value_base: (53980, 48)
events: (1800464, 24)
tracking: (13529224, 10)
players: (1172, 8)


In [4]:
origin_sequences.head()

,chain_id,game_id,period,sequence_id,team_id,raw_first_chance_team_id,n_team_id_overrides_for_chain,chain_start_time,chain_end_time,chain_duration_seconds,...,first_followup_event_id,first_followup_time,followup_max_xg,followup_sum_xg,chain_goal,chain_goal_event_id,chain_goal_time,chain_goal_player_id,chain_goal_player_name,chain_any_goal_within_2s_flag
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,11,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,0,591.73,591.73,0.0,...,<NA>,NaN,0.0,0.0,False,<NA>,NaN,NaN,NaN,False
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,722.40,722.40,0.0,...,<NA>,NaN,0.0,0.0,False,<NA>,NaN,NaN,NaN,False
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,736.33,736.33,0.0,...,<NA>,NaN,0.0,0.0,False,<NA>,NaN,NaN,NaN,False
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,765.73,765.73,0.0,...,<NA>,NaN,0.0,0.0,False,<NA>,NaN,NaN,NaN,False
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s13_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,13,6cac12e2-0546-2c1a-689f-ab26d8a6355a,6cac12e2-0546-2c1a-689f-ab26d8a6355a,0,923.80,923.80,0.0,...,<NA>,NaN,0.0,0.0,False,<NA>,NaN,NaN,NaN,False


In [5]:
origin_sequences.info()

<class 'pandas.DataFrame'>
RangeIndex: 48673 entries, 0 to 48672
Data columns (total 43 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   chain_id                         48673 non-null  str    
 1   game_id                          48673 non-null  str    
 2   period                           48673 non-null  int64  
 3   sequence_id                      48673 non-null  int64  
 4   team_id                          48673 non-null  str    
 5   raw_first_chance_team_id         48673 non-null  str    
 6   n_team_id_overrides_for_chain    48673 non-null  int64  
 7   chain_start_time                 48673 non-null  float64
 8   chain_end_time                   48673 non-null  float64
 9   chain_duration_seconds           48673 non-null  float64
 10  n_chances_in_chain               48673 non-null  int64  
 11  anchor_origin_event_id           48671 non-null  Int64  
 12  anchor_origin_period_time    

## 3. Validate Required Fields

The tracking join depends on:

- `game_id`
- `anchor_origin_event_id`
- `team_id`

The `anchor_origin_event_id` is the event ID for the origin shot release. This is the tracking snapshot we want.

In [6]:
required_origin_columns = [
    "chain_id",
    "game_id",
    "period",
    "sequence_id",
    "team_id",
    "anchor_origin_event_id",
    "anchor_origin_period_time",
    "anchor_origin_location",
]

origin_sequences[required_origin_columns].isna().sum()

chain_id                     0
game_id                      0
period                       0
sequence_id                  0
team_id                      0
anchor_origin_event_id       2
anchor_origin_period_time    2
anchor_origin_location       2
dtype: int64

In [7]:
# Event IDs should be unique within game, which lets us join tracking on game_id + sl_event_id.

events.duplicated(["game_id", "sl_event_id"]).sum()

np.int64(0)

In [8]:
# Tracking has one row per tracked player per event, so game_id + sl_event_id is not unique.
# That is expected.

tracking.duplicated(["game_id", "sl_event_id"]).sum()

np.int64(11902817)

## 4. Create Origin Event Table

We create one row per chain with a valid origin event.

The origin event is where the shot decision happened. For normal shots, this is the shot row. For deflections, this is the linked original shot row.

In [9]:
origin_events = origin_sequences[
    origin_sequences["anchor_origin_event_id"].notna()
].copy()

origin_events["anchor_origin_event_id"] = origin_events["anchor_origin_event_id"].astype("Int64")

origin_events = origin_events[
    [
        "chain_id",
        "game_id",
        "period",
        "sequence_id",
        "team_id",
        "anchor_origin_event_id",
        "anchor_origin_period_time",
        "anchor_origin_location",
        "first_evaluated_event_type",
        "first_evaluated_xg",
        "chain_max_xg",
        "has_deflection",
        "has_followup_chance",
        "chain_goal",
    ]
].copy()

origin_events.shape

(48671, 14)

## 5. Attach Origin Event Coordinates and Orientation

The event table already contains `x_adj` and `y_adj`, which are coordinates adjusted so the attacking team moves toward positive x.

The tracking table contains raw `tracking_x` and `tracking_y`.

To convert tracking coordinates into the same attacking-perspective frame, we infer the event-level orientation sign from the origin event:

- if `x_adj` matches `x`, orientation sign is `+1`
- if `x_adj` matches `-x`, orientation sign is `-1`

Then:

- `tracking_x_adj = tracking_x * orientation_sign`
- `tracking_y_adj = tracking_y * orientation_sign`

We validate this before using it.

In [10]:
origin_event_rows = events[
    [
        "game_id",
        "sl_event_id",
        "period",
        "period_time",
        "team_id",
        "player_id",
        "player_name",
        "event_type",
        "x",
        "y",
        "x_adj",
        "y_adj",
        "has_tracking_data",
        "event_player_tracked",
    ]
].copy()

origin_event_rows = origin_event_rows.rename(
    columns={
        "sl_event_id": "anchor_origin_event_id",
        "period_time": "origin_event_period_time",
        "team_id": "origin_event_team_id",
        "player_id": "origin_shooter_id",
        "player_name": "origin_shooter_name",
        "event_type": "origin_event_type",
        "x": "origin_x",
        "y": "origin_y",
        "x_adj": "origin_x_adj",
        "y_adj": "origin_y_adj",
        "has_tracking_data": "origin_has_tracking_data",
        "event_player_tracked": "origin_event_player_tracked",
    }
)

origin_events = origin_events.merge(
    origin_event_rows,
    how="left",
    on=["game_id", "anchor_origin_event_id"],
)

origin_events.shape

(48671, 26)

In [11]:
# Validate that the origin event row matches the chain's team and period.

origin_event_validation = {
    "missing_origin_event_rows": origin_events["origin_event_type"].isna().sum(),
    "period_mismatches": (origin_events["period_x"] != origin_events["period_y"]).sum(),
    "team_mismatches": (origin_events["team_id"] != origin_events["origin_event_team_id"]).sum(),
}

origin_event_validation

{'missing_origin_event_rows': np.int64(0),
 'period_mismatches': np.int64(0),
 'team_mismatches': np.int64(0)}

In [12]:
# Infer attacking orientation from origin event x and x_adj.
#
# Most rows should have origin_x_adj approximately equal to either origin_x or -origin_x.
# Near-zero x values can be ambiguous, so we use y/y_adj as a fallback if needed.

def infer_orientation_sign(row):
    x = row["origin_x"]
    x_adj = row["origin_x_adj"]
    y = row["origin_y"]
    y_adj = row["origin_y_adj"]

    if pd.notna(x) and pd.notna(x_adj) and abs(x) > 1e-6:
        return 1 if abs(x_adj - x) <= abs(x_adj + x) else -1

    if pd.notna(y) and pd.notna(y_adj) and abs(y) > 1e-6:
        return 1 if abs(y_adj - y) <= abs(y_adj + y) else -1

    return pd.NA


origin_events["orientation_sign"] = origin_events.apply(infer_orientation_sign, axis=1)

origin_events["orientation_sign"].value_counts(dropna=False)

orientation_sign
 1    24485
-1    24186
Name: count, dtype: int64

In [13]:
# Check how well the inferred sign reconstructs adjusted event coordinates.

origin_events["x_adj_reconstructed"] = (
    origin_events["origin_x"] * origin_events["orientation_sign"].astype(float)
)
origin_events["y_adj_reconstructed"] = (
    origin_events["origin_y"] * origin_events["orientation_sign"].astype(float)
)

origin_events["x_adj_error"] = (
    origin_events["origin_x_adj"] - origin_events["x_adj_reconstructed"]
).abs()

origin_events["y_adj_error"] = (
    origin_events["origin_y_adj"] - origin_events["y_adj_reconstructed"]
).abs()

origin_events[["x_adj_error", "y_adj_error"]].describe()

,x_adj_error,y_adj_error
count,48671.0,48671.0
mean,0.0,0.0
std,0.0,0.0
min,0.0,0.0
25%,0.0,0.0
50%,0.0,0.0
75%,0.0,0.0
max,0.0,0.0


In [14]:
# Inspect worst reconstruction errors if any exist.

origin_events.sort_values(
    ["x_adj_error", "y_adj_error"],
    ascending=False,
).head(20)[
    [
        "chain_id",
        "game_id",
        "anchor_origin_event_id",
        "origin_event_type",
        "origin_x",
        "origin_y",
        "origin_x_adj",
        "origin_y_adj",
        "orientation_sign",
        "x_adj_reconstructed",
        "y_adj_reconstructed",
        "x_adj_error",
        "y_adj_error",
    ]
]

,chain_id,game_id,anchor_origin_event_id,origin_event_type,origin_x,origin_y,origin_x_adj,origin_y_adj,orientation_sign,x_adj_reconstructed,y_adj_reconstructed,x_adj_error,y_adj_error
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,565,shot,-32.889107,-25.397057,32.889107,25.397057,-1,32.889107,25.397057,0.0,0.0
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,687,shot,52.107956,-8.799999,52.107956,-8.799999,1,52.107956,-8.799999,0.0,0.0
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,702,shot,64.681470,-20.870590,64.681470,-20.870590,1,64.681470,-20.870590,0.0,0.0
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,730,shot,77.757950,5.785294,77.757950,5.785294,1,77.757950,5.785294,0.0,0.0
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s13_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,882,shot,57.640305,26.405882,57.640305,26.405882,1,57.640305,26.405882,0.0,0.0
5,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s13_td...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,746,shot,-62.562637,22.885294,62.562637,-22.885294,-1,62.562637,-22.885294,0.0,0.0
6,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s13_td...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,837,shot,-56.024403,-23.888237,56.024403,23.888237,-1,56.024403,23.888237,0.0,0.0
7,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s14_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,889,shot,41.546190,11.820587,41.546190,11.820587,1,41.546190,11.820587,0.0,0.0
8,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s15_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,973,shot,62.669710,-24.391174,62.669710,-24.391174,1,62.669710,-24.391174,0.0,0.0
9,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s15_td...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,933,shot,-79.662636,3.773529,79.662636,-3.773529,-1,79.662636,-3.773529,0.0,0.0


## 6. Join Tracking At Origin Event

Tracking is joined at the origin shot event:

- `game_id`
- `anchor_origin_event_id`

The tracking table has one row per tracked player per event, so this join intentionally expands the data to one row per chain-player at the origin shot moment.

After computing player-location features, we aggregate back to one row per chain.

In [15]:
# Prepare tracking table for origin-event join.

tracking_origin = tracking.rename(
    columns={
        "sl_event_id": "anchor_origin_event_id",
        "team_id": "tracking_team_id",
        "team_name": "tracking_team_name",
        "player_id": "tracking_player_id",
        "player_name": "tracking_player_name",
    }
).copy()

tracking_origin.head()

,game_id,anchor_origin_event_id,tracking_team_id,tracking_team_name,tracking_player_id,tracking_player_name,tracking_x,tracking_y,tracking_vel_x,tracking_vel_y
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,Monsters,0f052c34-6cd2-f4fd-a861-d9051a8e86e4,"Angle, Tyler",1.735564,-12.985565,NaN,NaN
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,6cac12e2-0546-2c1a-689f-ab26d8a6355a,Griffins,663df049-a045-0561-16e3-1db633a0723e,"Aston-Reese, Zachary",-2.552494,13.848426,NaN,NaN
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,6cac12e2-0546-2c1a-689f-ab26d8a6355a,Griffins,015f554a-21c0-99bc-0a31-a176810b40c6,"Shine, Dominik",-2.368766,-12.660762,NaN,NaN
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,Monsters,NaN,NaN,0.216535,13.635171,NaN,NaN
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,6cac12e2-0546-2c1a-689f-ab26d8a6355a,Griffins,9cdb062f-6fc7-e205-1858-d4f6f32237a4,"Johansson, Albert",-20.072179,7.312992,NaN,NaN


In [16]:
# Join tracking rows to each origin shot event.
#
# This creates one row per tracked player at the origin shot moment.

origin_tracking = origin_events.merge(
    tracking_origin,
    how="left",
    on=["game_id", "anchor_origin_event_id"],
)

origin_tracking.shape

(417875, 39)

In [17]:
# Tracking rows per chain.

tracking_rows_per_chain = (
    origin_tracking
    .groupby("chain_id")
    .size()
    .rename("tracking_rows")
    .reset_index()
)

tracking_rows_per_chain["tracking_rows"].describe()

count    48671.000000
mean         8.585708
std          2.660598
min          1.000000
25%          8.000000
50%          9.000000
75%         10.000000
max         15.000000
Name: tracking_rows, dtype: float64

## 7. Adjust Tracking Coordinates To Attacking Perspective

The event-level orientation sign was validated above.

We now transform tracking coordinates into the same offensive-frame coordinate system used by `x_adj` and `y_adj`.

In [18]:
# Convert tracking coordinates into attacking-perspective coordinates.

origin_tracking["tracking_x_adj"] = (
    origin_tracking["tracking_x"] * origin_tracking["orientation_sign"].astype(float)
)

origin_tracking["tracking_y_adj"] = (
    origin_tracking["tracking_y"] * origin_tracking["orientation_sign"].astype(float)
)

origin_tracking[["tracking_x", "tracking_y", "tracking_x_adj", "tracking_y_adj"]].head()

,tracking_x,tracking_y,tracking_x_adj,tracking_y_adj
0,-60.091865,-17.772310,60.091865,17.772310
1,-77.654202,-5.987533,77.654202,5.987533
2,-30.616799,-26.758531,30.616799,26.758531
3,-75.246065,-4.412730,75.246065,4.412730
4,-46.620736,-11.062992,46.620736,11.062992


## 8. Add Player Position Metadata

Goalies should not be counted as skaters in slot-support features.

We use `players.parquet` to identify goalie rows where possible.

In [19]:
players.head()

,player_id,player_name,last_name,first_name,handed,birth_date,position_group,primary_position
0,58ff86bf-e659-f9dc-3f5f-6a51741e47ab,"Zohorna, Radim",Zohorna,Radim,L,1996-04-29,F,C
1,87bd2512-ca41-d650-0517-297bbfc26a41,"Philp, Luke",Philp,Luke,R,1995-11-06,F,C
2,cb889642-eedd-496a-9c02-563f569c6359,"Roos, Filip",Roos,Filip,L,1999-01-05,D,D
3,69c1e284-b220-e453-250a-66413a82618e,"Studenic, Marian",Studenic,Marian,L,1998-10-28,F,LW
4,f8513815-ad09-b742-fef8-9814f2313b80,"Steen, Oskar",Steen,Oskar,R,1998-03-09,F,RW


In [20]:
players[["position_group", "primary_position"]].value_counts(dropna=False)

position_group  primary_position
D               D                   371
F               F                   221
                C                   182
                LW                  157
                RW                  127
G               G                   114
Name: count, dtype: int64

In [21]:
# Build goalie flag from player metadata.

player_positions = players[
    [
        "player_id",
        "position_group",
        "primary_position",
    ]
].rename(
    columns={
        "player_id": "tracking_player_id",
    }
).copy()

player_positions["is_goalie"] = (
    player_positions["position_group"].astype(str).str.lower().str.contains("goal")
    | player_positions["primary_position"].astype(str).str.lower().isin(["g", "goalie", "goaltender"])
)

player_positions["is_goalie"].value_counts(dropna=False)

is_goalie
False    1058
True      114
Name: count, dtype: int64

In [22]:
origin_tracking = origin_tracking.merge(
    player_positions,
    how="left",
    on="tracking_player_id",
)

origin_tracking["is_goalie"] = origin_tracking["is_goalie"].fillna(False)

origin_tracking[
    [
        "tracking_player_id",
        "tracking_player_name",
        "position_group",
        "primary_position",
        "is_goalie",
    ]
].head()

,tracking_player_id,tracking_player_name,position_group,primary_position,is_goalie
0,cf9158fc-30e9-680c-b9e5-b65717e1e000,"Czarnik, Austin",F,C,False
1,c9060c85-0600-ed69-de2d-6aa7611fc7bc,"Gaunce, Brendan",F,C,False
2,cb8a6b8b-c5a6-4bfe-10b7-15c784b83a7d,"Bjork, Marcus",D,D,False
3,NaN,NaN,NaN,NaN,False
4,7a213ae8-292c-bc7f-83dd-7b14bdcadde1,"Berggren, Jonatan",F,RW,False


## 9. Define Slot Occupancy At Origin Shot

We use the same first-pass geometric slot definition as earlier notebooks:

- `tracking_x_adj >= 54`
- `abs(tracking_y_adj) <= 22`

This is a transparent rectangular proxy. Later analysis can test sensitivity to alternate slot definitions.

Slot-support counts are measured at the origin shot event.

For attacking skaters, the origin shooter is excluded.

In [23]:
SLOT_X_MIN = 54
SLOT_ABS_Y_MAX = 22

origin_tracking["tracking_abs_y_adj"] = origin_tracking["tracking_y_adj"].abs()

origin_tracking["tracking_in_slot"] = (
    (origin_tracking["tracking_x_adj"] >= SLOT_X_MIN)
    & (origin_tracking["tracking_abs_y_adj"] <= SLOT_ABS_Y_MAX)
)

origin_tracking["is_attacking_team"] = (
    origin_tracking["tracking_team_id"] == origin_tracking["team_id"]
)

origin_tracking["is_defending_team"] = (
    origin_tracking["tracking_team_id"].notna()
    & (origin_tracking["tracking_team_id"] != origin_tracking["team_id"])
)

origin_tracking["is_origin_shooter"] = (
    origin_tracking["tracking_player_id"] == origin_tracking["origin_shooter_id"]
)

origin_tracking[
    [
        "tracking_player_name",
        "tracking_team_id",
        "team_id",
        "is_attacking_team",
        "is_defending_team",
        "is_origin_shooter",
        "is_goalie",
        "tracking_x_adj",
        "tracking_y_adj",
        "tracking_in_slot",
    ]
].head()

,tracking_player_name,tracking_team_id,team_id,is_attacking_team,is_defending_team,is_origin_shooter,is_goalie,tracking_x_adj,tracking_y_adj,tracking_in_slot
0,"Czarnik, Austin",6cac12e2-0546-2c1a-689f-ab26d8a6355a,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,False,True,False,False,60.091865,17.772310,True
1,"Gaunce, Brendan",d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,True,False,False,False,77.654202,5.987533,True
2,"Bjork, Marcus",d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,True,False,True,False,30.616799,26.758531,False
3,NaN,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,True,False,False,False,75.246065,4.412730,True
4,"Berggren, Jonatan",6cac12e2-0546-2c1a-689f-ab26d8a6355a,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,False,True,False,False,46.620736,11.062992,False


In [24]:
# Basic sanity checks for team/player flags.

flag_checks = {
    "tracking_rows": len(origin_tracking),
    "missing_tracking_player_id": origin_tracking["tracking_player_id"].isna().sum(),
    "attacking_rows": int(origin_tracking["is_attacking_team"].sum()),
    "defending_rows": int(origin_tracking["is_defending_team"].sum()),
    "origin_shooter_rows": int(origin_tracking["is_origin_shooter"].sum()),
    "goalie_rows": int(origin_tracking["is_goalie"].sum()),
}

flag_checks

{'tracking_rows': 417875,
 'missing_tracking_player_id': np.int64(134896),
 'attacking_rows': 177255,
 'defending_rows': 237361,
 'origin_shooter_rows': 33539,
 'goalie_rows': 3503}

## 10. Aggregate Observed Slot-Support Features

Aggregate back to one row per chain.

Primary features:

- observed identified attacking skaters in slot at origin, excluding shooter and goalies
- observed identified defending skaters in slot at origin, excluding goalies
- identified tracking player count
- tracking availability and diagnostic/error flags

Tracking completeness is not the same thing as game state. True 5v5 status is added from `stints.parquet` in the next notebook.

In [25]:
# Player-count features at origin shot release.
#
# Important:
# - Only identified player rows can count as skaters.
# - Missing-player tracking rows are retained for diagnostics but excluded from skater counts.
# - Aggregation counts unique player IDs, not tracking rows.
# - Slot counts are observed counts, not guaranteed full on-ice occupancy.

origin_tracking["has_tracking_player_id"] = origin_tracking["tracking_player_id"].notna()

origin_tracking["origin_tracking_row_present"] = (
    origin_tracking["tracking_x"].notna()
    & origin_tracking["tracking_y"].notna()
)

origin_tracking["origin_missing_player_tracking_row"] = (
    origin_tracking["origin_tracking_row_present"]
    & origin_tracking["tracking_player_id"].isna()
)

origin_tracking["attacker_slot_player_id"] = np.where(
    origin_tracking["has_tracking_player_id"]
    & origin_tracking["is_attacking_team"]
    & ~origin_tracking["is_origin_shooter"]
    & ~origin_tracking["is_goalie"]
    & origin_tracking["tracking_in_slot"],
    origin_tracking["tracking_player_id"],
    pd.NA,
)

origin_tracking["defender_slot_player_id"] = np.where(
    origin_tracking["has_tracking_player_id"]
    & origin_tracking["is_defending_team"]
    & ~origin_tracking["is_goalie"]
    & origin_tracking["tracking_in_slot"],
    origin_tracking["tracking_player_id"],
    pd.NA,
)

origin_tracking["attacker_skater_player_id"] = np.where(
    origin_tracking["has_tracking_player_id"]
    & origin_tracking["is_attacking_team"]
    & ~origin_tracking["is_goalie"],
    origin_tracking["tracking_player_id"],
    pd.NA,
)

origin_tracking["defender_skater_player_id"] = np.where(
    origin_tracking["has_tracking_player_id"]
    & origin_tracking["is_defending_team"]
    & ~origin_tracking["is_goalie"],
    origin_tracking["tracking_player_id"],
    pd.NA,
)

origin_tracking["goalie_player_id"] = np.where(
    origin_tracking["has_tracking_player_id"]
    & origin_tracking["is_goalie"],
    origin_tracking["tracking_player_id"],
    pd.NA,
)

slot_support_features = (
    origin_tracking
    .groupby("chain_id")
    .agg(
        origin_tracking_rows=("origin_tracking_row_present", "sum"),
        origin_identified_tracking_players=("tracking_player_id", "nunique"),
        origin_missing_tracking_player_rows=("origin_missing_player_tracking_row", "sum"),
        origin_attacker_skaters_tracked=("attacker_skater_player_id", "nunique"),
        origin_defender_skaters_tracked=("defender_skater_player_id", "nunique"),
        observed_origin_attackers_in_slot=("attacker_slot_player_id", "nunique"),
        observed_origin_defenders_in_slot=("defender_slot_player_id", "nunique"),
        origin_goalies_tracked=("goalie_player_id", "nunique"),
        origin_shooter_tracked=("is_origin_shooter", "max"),
    )
    .reset_index()
)

slot_support_features.head()

,chain_id,origin_tracking_rows,origin_identified_tracking_players,origin_missing_tracking_player_rows,origin_attacker_skaters_tracked,origin_defender_skaters_tracked,observed_origin_attackers_in_slot,observed_origin_defenders_in_slot,origin_goalies_tracked,origin_shooter_tracked
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,8,7,1,4,3,1,2,0,True
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,10,7,3,4,3,1,2,0,True
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,8,5,3,3,2,0,1,0,True
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,8,4,4,1,3,1,3,1,False
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s13_t6...,5,3,2,2,1,0,1,0,True


In [26]:
# Add tracking availability, completeness, and error diagnostics.
#
# These are diagnostics, not the primary hockey cohort filter.
# True 5v5 status is added from stints in the next notebook.

slot_support_features["origin_tracking_available"] = (
    slot_support_features["origin_identified_tracking_players"] > 0
)

slot_support_features["origin_tracking_error_flag"] = (
    (slot_support_features["origin_attacker_skaters_tracked"] > 6)
    | (slot_support_features["origin_defender_skaters_tracked"] > 6)
    | (slot_support_features["origin_goalies_tracked"] > 2)
)

slot_support_features["origin_tracking_completeness_bucket"] = np.select(
    [
        slot_support_features["origin_tracking_error_flag"],
        slot_support_features["origin_identified_tracking_players"].eq(0),
        (slot_support_features["origin_attacker_skaters_tracked"] >= 5)
        & (slot_support_features["origin_defender_skaters_tracked"] >= 5),
        (slot_support_features["origin_attacker_skaters_tracked"] >= 4)
        & (slot_support_features["origin_defender_skaters_tracked"] >= 4),
        slot_support_features["origin_identified_tracking_players"].gt(0),
    ],
    [
        "tracking_error_flagged",
        "no_identified_players",
        "near_full_observed",
        "high_observed",
        "partial_observed",
    ],
    default="no_identified_players",
)

slot_support_features["origin_tracking_completeness_bucket"].value_counts(dropna=False)

origin_tracking_completeness_bucket
partial_observed          37255
high_observed              7136
no_identified_players      3328
near_full_observed          929
tracking_error_flagged       23
Name: count, dtype: int64

In [27]:
# Distribution of observed slot-support features.

slot_support_features[
    [
        "origin_tracking_rows",
        "origin_identified_tracking_players",
        "origin_missing_tracking_player_rows",
        "origin_attacker_skaters_tracked",
        "origin_defender_skaters_tracked",
        "observed_origin_attackers_in_slot",
        "observed_origin_defenders_in_slot",
        "origin_goalies_tracked",
        "origin_shooter_tracked",
        "origin_tracking_available",
        "origin_tracking_error_flag",
        "origin_tracking_completeness_bucket",
    ]
].describe(include="all")

,origin_tracking_rows,origin_identified_tracking_players,origin_missing_tracking_player_rows,origin_attacker_skaters_tracked,origin_defender_skaters_tracked,observed_origin_attackers_in_slot,observed_origin_defenders_in_slot,origin_goalies_tracked,origin_shooter_tracked,origin_tracking_available,origin_tracking_error_flag,origin_tracking_completeness_bucket
count,48671.000000,48671.000000,48671.000000,48671.000000,48671.000000,48671.000000,48671.000000,48671.000000,48671,48671,48671,48671
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,2,2,5
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,True,False,partial_observed
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33539,45343,48648,37255
mean,8.518748,5.814119,2.704629,2.718107,3.096012,0.905550,1.953340,0.071973,NaN,NaN,NaN,NaN
std,2.856073,2.389164,1.688780,1.431057,1.424436,0.880919,1.337615,0.258446,NaN,NaN,NaN,NaN
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
25%,8.000000,5.000000,2.000000,2.000000,2.000000,0.000000,1.000000,0.000000,NaN,NaN,NaN,NaN
50%,9.000000,6.000000,3.000000,3.000000,3.000000,1.000000,2.000000,0.000000,NaN,NaN,NaN,NaN
75%,10.000000,7.000000,4.000000,4.000000,4.000000,2.000000,3.000000,0.000000,NaN,NaN,NaN,NaN


## 11. Audit Tracking Row Semantics

These diagnostics document why the final aggregation counts unique identified player IDs rather than row-level booleans.

The tracking table includes unidentified rows with team/location fields but no player ID. Those rows are useful for data-quality diagnostics, but they cannot be counted as skaters.

Incomplete tracking does not automatically invalidate a chain. It means the slot-support variables should be interpreted as **observed identified players in the slot**, not full on-ice occupancy.

In [28]:
tracking.columns

Index(['game_id', 'sl_event_id', 'team_id', 'team_name', 'player_id',
       'player_name', 'tracking_x', 'tracking_y', 'tracking_vel_x',
       'tracking_vel_y'],
      dtype='str')

In [29]:
tracking.head(20)

,game_id,sl_event_id,team_id,team_name,player_id,player_name,tracking_x,tracking_y,tracking_vel_x,tracking_vel_y
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,Monsters,0f052c34-6cd2-f4fd-a861-d9051a8e86e4,"Angle, Tyler",1.735564,-12.985565,NaN,NaN
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,6cac12e2-0546-2c1a-689f-ab26d8a6355a,Griffins,663df049-a045-0561-16e3-1db633a0723e,"Aston-Reese, Zachary",-2.552494,13.848426,NaN,NaN
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,6cac12e2-0546-2c1a-689f-ab26d8a6355a,Griffins,015f554a-21c0-99bc-0a31-a176810b40c6,"Shine, Dominik",-2.368766,-12.660762,NaN,NaN
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,Monsters,NaN,NaN,0.216535,13.635171,NaN,NaN
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,6cac12e2-0546-2c1a-689f-ab26d8a6355a,Griffins,9cdb062f-6fc7-e205-1858-d4f6f32237a4,"Johansson, Albert",-20.072179,7.312992,NaN,NaN
5,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,6cac12e2-0546-2c1a-689f-ab26d8a6355a,Griffins,NaN,NaN,-2.237533,10.006562,0.688983,-6.692981
6,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,6cac12e2-0546-2c1a-689f-ab26d8a6355a,Griffins,NaN,NaN,-18.766405,-8.041339,-0.393705,0.885836
7,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,Monsters,0753b094-9e2f-976d-85c8-d22a1d280e8d,"Jiricek, David",18.441602,17.211287,NaN,NaN
8,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,6cac12e2-0546-2c1a-689f-ab26d8a6355a,Griffins,8cdcb61e-d733-bde6-a101-ef3140e48149,"L'Esperance, Joel",-2.362205,0.695538,NaN,NaN
9,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,Monsters,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",1.099081,-0.456037,NaN,NaN


In [30]:
# Summarize missing-player rows.
#
# Important distinction:
# - unmatched origin events from a left join have no tracking coordinates
# - unidentified tracked objects can have coordinates/team IDs but no player ID
#
# Only the second type is evidence of real tracking rows without player identity.

origin_tracking["origin_tracking_row_present"] = (
    origin_tracking["tracking_x"].notna()
    & origin_tracking["tracking_y"].notna()
)

origin_tracking["origin_missing_player_tracking_row"] = (
    origin_tracking["origin_tracking_row_present"]
    & origin_tracking["tracking_player_id"].isna()
)

missing_player_tracking_rows = origin_tracking[
    origin_tracking["origin_missing_player_tracking_row"]
].copy()

missing_player_summary = {
    "missing_player_tracking_rows": len(missing_player_tracking_rows),
    "missing_tracking_team_id": missing_player_tracking_rows["tracking_team_id"].isna().sum(),
    "missing_tracking_team_name": missing_player_tracking_rows["tracking_team_name"].isna().sum()
        if "tracking_team_name" in missing_player_tracking_rows.columns else "column_missing",
    "missing_tracking_x": missing_player_tracking_rows["tracking_x"].isna().sum(),
    "missing_tracking_y": missing_player_tracking_rows["tracking_y"].isna().sum(),
}

missing_player_summary

{'missing_player_tracking_rows': 131637,
 'missing_tracking_team_id': np.int64(0),
 'missing_tracking_team_name': np.int64(0),
 'missing_tracking_x': np.int64(0),
 'missing_tracking_y': np.int64(0)}

In [31]:
# Are player rows duplicated within the same chain?

player_row_dupes = (
    origin_tracking[
        origin_tracking["tracking_player_id"].notna()
    ]
    .groupby(["chain_id", "tracking_player_id"])
    .size()
    .rename("rows_for_player_chain")
    .reset_index()
)

player_row_dupes["rows_for_player_chain"].value_counts().sort_index().head(20)

rows_for_player_chain
1    282979
Name: count, dtype: int64

In [32]:
# Position metadata coverage among tracked player rows.

position_coverage = {
    "tracked_player_rows": origin_tracking["tracking_player_id"].notna().sum(),
    "missing_position_group_rows": origin_tracking.loc[
        origin_tracking["tracking_player_id"].notna(), "position_group"
    ].isna().sum(),
    "missing_primary_position_rows": origin_tracking.loc[
        origin_tracking["tracking_player_id"].notna(), "primary_position"
    ].isna().sum(),
}

position_coverage

{'tracked_player_rows': np.int64(282979),
 'missing_position_group_rows': np.int64(0),
 'missing_primary_position_rows': np.int64(0)}

In [33]:
# Distinct player counts per chain/team, using unique player IDs rather than row sums.

distinct_team_counts = (
    origin_tracking[
        origin_tracking["tracking_player_id"].notna()
    ]
    .assign(
        tracked_attacker_skaters_id=lambda d: np.where(
            d["is_attacking_team"] & ~d["is_goalie"],
            d["tracking_player_id"],
            pd.NA,
        ),
        tracked_defender_skaters_id=lambda d: np.where(
            d["is_defending_team"] & ~d["is_goalie"],
            d["tracking_player_id"],
            pd.NA,
        ),
        tracked_goalie_id=lambda d: np.where(
            d["is_goalie"],
            d["tracking_player_id"],
            pd.NA,
        ),
    )
    .groupby("chain_id")
    .agg(
        unique_tracking_players=("tracking_player_id", "nunique"),
        unique_attacker_skaters=("tracked_attacker_skaters_id", "nunique"),
        unique_defender_skaters=("tracked_defender_skaters_id", "nunique"),
        unique_goalies=("tracked_goalie_id", "nunique"),
    )
    .reset_index()
)

distinct_team_counts.describe()

,unique_tracking_players,unique_attacker_skaters,unique_defender_skaters,unique_goalies
count,45343.000000,45343.000000,45343.000000,45343.000000
mean,6.240853,2.917606,3.323247,0.077256
std,1.861135,1.271285,1.192799,0.266999
min,1.000000,0.000000,0.000000,0.000000
25%,5.000000,2.000000,3.000000,0.000000
50%,6.000000,3.000000,3.000000,0.000000
75%,8.000000,4.000000,4.000000,0.000000
max,12.000000,7.000000,7.000000,1.000000


In [34]:
# Find impossible or suspicious unique counts.

suspicious_counts = distinct_team_counts[
    (distinct_team_counts["unique_attacker_skaters"] > 6)
    | (distinct_team_counts["unique_defender_skaters"] > 6)
    | (distinct_team_counts["unique_goalies"] > 2)
].copy()

suspicious_counts.shape

(23, 5)

In [35]:
# Inspect tracking-error-flag cases.
# These are retained but flagged.

slot_support_features[
    slot_support_features["origin_tracking_error_flag"]
].sort_values(
    ["origin_attacker_skaters_tracked", "origin_defender_skaters_tracked"],
    ascending=False,
).head(25)

,chain_id,origin_tracking_rows,origin_identified_tracking_players,origin_missing_tracking_player_rows,origin_attacker_skaters_tracked,origin_defender_skaters_tracked,observed_origin_attackers_in_slot,observed_origin_defenders_in_slot,origin_goalies_tracked,origin_shooter_tracked,origin_tracking_available,origin_tracking_error_flag,origin_tracking_completeness_bucket
27363,9acc5269-b561-2877-6b79-f90087c23e57_p2_s25_te...,13,12,1,7,5,2,4,0,True,True,True,tracking_error_flagged
28817,a2466f78-d8ee-a3c7-fe6f-fa9076aad73b_p3_s51_t8...,14,11,3,7,4,3,4,0,True,True,True,tracking_error_flagged
755,06795c32-cffd-e9e8-3a6a-5728a7f07a8b_p1_s2_t9c...,11,10,1,7,3,3,1,0,True,True,True,tracking_error_flagged
2933,16581738-b8d5-e8b6-fa0d-ad9b41589d03_p2_s24_te...,12,10,2,7,3,3,3,0,True,True,True,tracking_error_flagged
47125,faf6eb06-a5c4-b794-cee6-f23fca7751b6_p1_s1_tdc...,11,10,1,7,3,0,0,0,True,True,True,tracking_error_flagged
30592,ad636af9-1037-6f76-9f60-672619ad2b45_p2_s22_t0...,15,9,6,7,2,1,1,0,True,True,True,tracking_error_flagged
6121,24ece857-df50-dd83-d393-380132a72cdb_p3_s53_t2...,13,7,6,7,0,1,0,0,True,True,True,tracking_error_flagged
37254,c9dddad9-a5b2-958d-d588-3e3ac8a3c41b_p2_s42_t8...,14,12,2,5,7,1,3,0,True,True,True,tracking_error_flagged
1681,0c71a655-49c2-ec7a-64f1-3c33b7b31259_p1_s13_t8...,12,11,1,4,7,2,5,1,True,True,True,tracking_error_flagged
10318,408f0d44-b7bd-184a-6281-7985be9df008_p3_s44_t0...,13,11,2,4,7,2,3,0,True,True,True,tracking_error_flagged


In [36]:
# Hard validation: observed slot counts cannot exceed tracked skater counts.

slot_count_validation = {
    "attacker_slot_gt_attacker_tracked": (
        slot_support_features["observed_origin_attackers_in_slot"]
        > slot_support_features["origin_attacker_skaters_tracked"]
    ).sum(),
    "defender_slot_gt_defender_tracked": (
        slot_support_features["observed_origin_defenders_in_slot"]
        > slot_support_features["origin_defender_skaters_tracked"]
    ).sum(),
    "tracking_error_flag_rows": slot_support_features["origin_tracking_error_flag"].sum(),
    "chains_with_no_identified_players": (
        slot_support_features["origin_identified_tracking_players"] == 0
    ).sum(),
}

slot_count_validation

{'attacker_slot_gt_attacker_tracked': np.int64(0),
 'defender_slot_gt_defender_tracked': np.int64(0),
 'tracking_error_flag_rows': np.int64(23),
 'chains_with_no_identified_players': np.int64(3328)}

The diagnostic checks showed that missing-player rows are present and often carry team/location fields, so row-level boolean sums are unsafe. Player-count features therefore count unique non-null player IDs only.

The final slot-support fields are intentionally named `observed_origin_attackers_in_slot` and `observed_origin_defenders_in_slot` because they count identified tracked players visible in the event-frame data. They should not be interpreted as a guaranteed full on-ice census.

## 12. Merge Tracking Features Back To Origin Shot Sequences

We merge the tracking-derived slot-support features back to the one-row-per-chain table.

Chains with invalid or missing origin tracking retain null tracking features and are labeled as missing tracking quality.

In [37]:
origin_sequences_with_tracking = origin_sequences.merge(
    slot_support_features,
    how="left",
    on="chain_id",
    validate="one_to_one",
)

origin_sequences_with_tracking.shape

(48673, 55)

In [38]:
merge_validation = {
    "origin_sequences_rows": len(origin_sequences),
    "merged_rows": len(origin_sequences_with_tracking),
    "row_difference": len(origin_sequences_with_tracking) - len(origin_sequences),
    "duplicate_chain_ids": origin_sequences_with_tracking.duplicated("chain_id").sum(),
    "chains_missing_tracking_completeness_bucket": origin_sequences_with_tracking["origin_tracking_completeness_bucket"].isna().sum(),
}

merge_validation

{'origin_sequences_rows': 48673,
 'merged_rows': 48673,
 'row_difference': 0,
 'duplicate_chain_ids': np.int64(0),
 'chains_missing_tracking_completeness_bucket': np.int64(2)}

In [39]:
tracking_feature_columns = [
    "origin_tracking_rows",
    "origin_identified_tracking_players",
    "origin_missing_tracking_player_rows",
    "origin_attacker_skaters_tracked",
    "origin_defender_skaters_tracked",
    "observed_origin_attackers_in_slot",
    "observed_origin_defenders_in_slot",
    "origin_goalies_tracked",
]

for col in tracking_feature_columns:
    origin_sequences_with_tracking[col] = origin_sequences_with_tracking[col].fillna(0).astype("Int64")

origin_sequences_with_tracking["origin_shooter_tracked"] = (
    origin_sequences_with_tracking["origin_shooter_tracked"]
    .fillna(False)
    .astype(bool)
)

origin_sequences_with_tracking["origin_tracking_available"] = (
    origin_sequences_with_tracking["origin_tracking_available"]
    .fillna(False)
    .astype(bool)
)

origin_sequences_with_tracking["origin_tracking_error_flag"] = (
    origin_sequences_with_tracking["origin_tracking_error_flag"]
    .fillna(False)
    .astype(bool)
)

origin_sequences_with_tracking["origin_tracking_completeness_bucket"] = (
    origin_sequences_with_tracking["origin_tracking_completeness_bucket"]
    .fillna("no_identified_players")
)

In [40]:
final_tracking_validation = {
    "rows": len(origin_sequences_with_tracking),
    "unique_chain_ids": origin_sequences_with_tracking["chain_id"].nunique(),
    "duplicate_chain_ids": origin_sequences_with_tracking.duplicated("chain_id").sum(),
    "missing_tracking_completeness_bucket": origin_sequences_with_tracking["origin_tracking_completeness_bucket"].isna().sum(),
    "attacker_slot_gt_attacker_tracked": (
        origin_sequences_with_tracking["observed_origin_attackers_in_slot"]
        > origin_sequences_with_tracking["origin_attacker_skaters_tracked"]
    ).sum(),
    "defender_slot_gt_defender_tracked": (
        origin_sequences_with_tracking["observed_origin_defenders_in_slot"]
        > origin_sequences_with_tracking["origin_defender_skaters_tracked"]
    ).sum(),
    "tracking_error_flag_rows": origin_sequences_with_tracking["origin_tracking_error_flag"].sum(),
    "chains_with_tracking_available": origin_sequences_with_tracking["origin_tracking_available"].sum(),
}

final_tracking_validation

{'rows': 48673,
 'unique_chain_ids': 48673,
 'duplicate_chain_ids': np.int64(0),
 'missing_tracking_completeness_bucket': np.int64(0),
 'attacker_slot_gt_attacker_tracked': np.int64(0),
 'defender_slot_gt_defender_tracked': np.int64(0),
 'tracking_error_flag_rows': np.int64(23),
 'chains_with_tracking_available': np.int64(45343)}

In [41]:
tracking_summary_by_origin = (
    origin_sequences_with_tracking
    .groupby(["anchor_origin_location", "origin_tracking_completeness_bucket"], dropna=False)
    .agg(
        chains=("chain_id", "count"),
        mean_observed_attackers_in_slot=("observed_origin_attackers_in_slot", "mean"),
        median_observed_attackers_in_slot=("observed_origin_attackers_in_slot", "median"),
        mean_observed_defenders_in_slot=("observed_origin_defenders_in_slot", "mean"),
        median_observed_defenders_in_slot=("observed_origin_defenders_in_slot", "median"),
        mean_chain_max_xg=("chain_max_xg", "mean"),
        goal_rate=("chain_goal", "mean"),
    )
    .reset_index()
    .sort_values(["anchor_origin_location", "origin_tracking_completeness_bucket"])
)

tracking_summary_by_origin

,anchor_origin_location,origin_tracking_completeness_bucket,chains,mean_observed_attackers_in_slot,median_observed_attackers_in_slot,mean_observed_defenders_in_slot,median_observed_defenders_in_slot,mean_chain_max_xg,goal_rate
0,outside,high_observed,4948,1.510509,2.0,2.662086,3.0,0.033411,0.034762
1,outside,near_full_observed,708,1.819209,2.0,3.261299,3.0,0.036684,0.028249
2,outside,no_identified_players,1931,0.0,0.0,0.0,0.0,0.032387,0.039876
3,outside,partial_observed,22981,0.813977,1.0,1.665463,2.0,0.028787,0.030503
4,outside,tracking_error_flagged,14,1.071429,1.0,2.857143,3.0,0.064348,0.071429
5,slot,high_observed,2188,1.575411,2.0,3.431901,4.0,0.085819,0.098263
6,slot,near_full_observed,221,1.719457,2.0,3.954751,4.0,0.087885,0.095023
7,slot,no_identified_players,1397,0.0,0.0,0.0,0.0,0.094826,0.118826
8,slot,partial_observed,14274,0.893583,1.0,2.301807,2.0,0.093401,0.097520
9,slot,tracking_error_flagged,9,1.0,1.0,4.111111,4.0,0.097419,0.222222


In [42]:
output_path = DATA_PROCESSED / "origin_shot_sequences_with_tracking.parquet"

origin_sequences_with_tracking.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Shape:", origin_sequences_with_tracking.shape)

Saved: c:\Users\rinal\hockey-analytics\outside-shot-value\data\processed\origin_shot_sequences_with_tracking.parquet
Shape: (48673, 55)
